# 第四版后续启动（补充入口）

这是已通过板测的第四版 r1 的启动补充，不是新的算法或硬件版本。只读取最近的本版本报告，不重新运行长时间测速。

将本文件上传到 `/home/xilinx/jupyter_notebooks/gesture_order_accel_v4/`，新开本文件的 Python 3 内核，依次运行下面第1、2部分。保留原 Notebook、模型、硬件和 results 目录。

若旧服务器仍在运行：先在笔记本窗口 Ctrl+C，再在原板端 Notebook 点击停止一次，等 `ORDER_BOARD_SERVER_STOPPED` 后关闭旧内核。不要同时启动两个服务器。


## 1．初始化并检查

完成后应显示 `V4_PREFLIGHT_OK`；本部分不下载比特流。


In [ ]:
V4_RESTART_READY = False
from pathlib import Path
import os,sys
ROOT=Path('/home/xilinx/jupyter_notebooks')
APP=ROOT/'gesture_order_accel_v4'
for name in ('board_server','order_core','session_guard','setup_runtime','v3_runtime','v4_runtime','pipeline_runtime','validate_v4','start_v4'):
    module=sys.modules.get(name)
    if module is not None and Path(module.__file__).resolve()!=(APP/(name+'.py')).resolve():
        raise RuntimeError('当前内核缓存了其他版本的 '+name+'，请重启本文件内核再从第1部分运行。')
sys.path.insert(0,str(APP));os.chdir(APP)
from setup_runtime import prepare
prepare(ROOT)

from start_v4 import preflight
preflight(ROOT)
V4_RESTART_READY = True


## 2．读取已有报告并启动

看到 `ORDER_BOARD_SERVER_READY port=8765` 后，再运行笔记本端的 `start_laptop.bat`。本框持续显示 `[*]` 是正常状态。

冻结交接版的配置是 `serial_cv1_t3`。如果显示其他配置或报错，请先核对；不要删除失败报告或改哈希绕过检查。正式启动会重新加载同一套 FPGA v2 并产生新的运行日志，不会覆盖原测速报告。


In [ ]:
from pathlib import Path
ROOT = Path('/home/xilinx/jupyter_notebooks')
if not globals().get('V4_RESTART_READY', False):
    raise RuntimeError('请先在本内核运行本文件第1部分，等待 V4_PREFLIGHT_OK。')
from start_v4 import load_last_report, run_selected
v4_report = load_last_report()
run_selected(v4_report, ROOT)


## 3．停止后导出（需要保存本次新日志时）

先在笔记本窗口 Ctrl+C，再在本 Notebook 顶部点击停止一次，等待 `ORDER_BOARD_SERVER_STOPPED`，然后运行下框。若提示线程未退出，先保留提示，不要再次加载硬件。

本次新日志与已冻结的提交证据分别保存，不用新空会话覆盖原来的成功演示记录。


In [ ]:
from pathlib import Path
ROOT = Path('/home/xilinx/jupyter_notebooks')
from validate_v4 import export_results
from IPython.display import HTML, display
from urllib.parse import quote
archive = export_results()
href = '/files/' + quote(archive.relative_to(ROOT).as_posix(), safe='/')
display(HTML('<a href="' + href + '" download>下载本次第四版运行日志</a>'))
